In [1]:
from google.colab import drive

drive.mount("/content/gdrive")

Mounted at /content/gdrive


CONFLICT-01 — Initialize

In [2]:
import os
import json
import pandas as pd
import numpy as np

PROJECT_ROOT = "/content/gdrive/MyDrive/CrisisWeave_Project"

CONFLICT_RESULTS_DIR = os.path.join(
    PROJECT_ROOT,
    "04_Results",
    "Conflict"
)

os.makedirs(CONFLICT_RESULTS_DIR, exist_ok=True)

print("CrisisWeave Conflict Detection initialized.")
print("Output directory:", CONFLICT_RESULTS_DIR)

CrisisWeave Conflict Detection initialized.
Output directory: /content/gdrive/MyDrive/CrisisWeave_Project/04_Results/Conflict


CONFLICT-02 — Load unified evidence

In [3]:
evidence_path = os.path.join(
    PROJECT_ROOT,
    "04_Results",
    "Evidence",
    "CrisisWeave_Evidence_CW001_EV001.json"
)

with open(evidence_path, "r", encoding="utf-8") as f:
    evidence_record = json.load(f)

print("Evidence loaded successfully.")
print("Incident:", evidence_record["incident_id"])
print("Evidence:", evidence_record["evidence_id"])

Evidence loaded successfully.
Incident: CW-001
Evidence: EV-001


CONFLICT-03 — Extract comparable claims

In [4]:
claims = {
    "cnn_severity": evidence_record[
        "image_evidence"
    ]["cnn_severity"],

    "temporal_latest_severity": evidence_record[
        "temporal_evidence"
    ].get("latest_severity"),

    "temporal_trend": evidence_record[
        "temporal_evidence"
    ]["overall_temporal_trend"],

    "bert_category": evidence_record[
        "text_evidence"
    ]["bert_category"],

    "water_relationship": evidence_record[
        "gis_evidence"
    ]["water_relationship"],

    "response_status": evidence_record[
        "response_evidence"
    ]["response_status"]
}

print(json.dumps(
    claims,
    indent=2,
    ensure_ascii=False
))

{
  "cnn_severity": "High",
  "temporal_latest_severity": null,
  "temporal_trend": "INCREASING",
  "bert_category": "requests_or_urgent_needs",
  "water_relationship": null,
  "response_status": "En Route"
}


CONFLICT-04 — Severity conflict detector

In [5]:
severity_order = {
    "Low": 1,
    "Moderate": 2,
    "High": 3
}


def detect_severity_conflict(cnn_severity, temporal_severity):
    """
    Compare CNN severity with the latest temporal severity.
    """

    if (
        cnn_severity not in severity_order
        or temporal_severity not in severity_order
    ):
        return {
            "conflict": False,
            "status": "INSUFFICIENT_EVIDENCE",
            "difference": None
        }

    cnn_score = severity_order[cnn_severity]
    temporal_score = severity_order[temporal_severity]

    difference = abs(
        cnn_score - temporal_score
    )

    if difference == 0:
        return {
            "conflict": False,
            "status": "AGREEMENT",
            "difference": 0
        }

    if difference == 1:
        return {
            "conflict": True,
            "status": "MINOR_CONFLICT",
            "difference": difference
        }

    return {
        "conflict": True,
        "status": "MAJOR_CONFLICT",
        "difference": difference
    }


severity_conflict = detect_severity_conflict(
    claims["cnn_severity"],
    claims["temporal_latest_severity"]
)

print(json.dumps(
    severity_conflict,
    indent=2
))

{
  "conflict": false,
  "status": "INSUFFICIENT_EVIDENCE",
  "difference": null
}


CONFLICT-05 — Temporal trend consistency

In [6]:
def detect_temporal_tension(
    severity,
    temporal_trend
):
    if severity not in severity_order:
        return {
            "tension": False,
            "status": "INSUFFICIENT_EVIDENCE"
        }

    if temporal_trend == "INCREASING":
        if severity in ["Moderate", "High"]:
            return {
                "tension": False,
                "status": "CONSISTENT"
            }

    if temporal_trend == "DECREASING":
        return {
            "tension": False,
            "status": "CONSISTENT"
        }

    if temporal_trend == "STABLE":
        return {
            "tension": False,
            "status": "CONSISTENT"
        }

    return {
        "tension": False,
        "status": "INSUFFICIENT_EVIDENCE"
    }


temporal_tension = detect_temporal_tension(
    claims["cnn_severity"],
    claims["temporal_trend"]
)

print(json.dumps(
    temporal_tension,
    indent=2
))

{
  "tension": false,
  "status": "CONSISTENT"
}


CONFLICT-06 — Humanitarian evidence consistency

In [7]:
humanitarian_categories = {
    "requests_or_urgent_needs",
    "displaced_people_and_evacuations",
    "injured_or_dead_people",
    "infrastructure_and_utility_damage",
    "missing_or_found_people"
}


def evaluate_humanitarian_support(category):
    if category in humanitarian_categories:
        return {
            "supports_disaster_assessment": True,
            "status": "SUPPORTING"
        }

    if category == "not_humanitarian":
        return {
            "supports_disaster_assessment": False,
            "status": "NON_HUMANITARIAN"
        }

    return {
        "supports_disaster_assessment": False,
        "status": "NEUTRAL"
    }


humanitarian_evidence = evaluate_humanitarian_support(
    claims["bert_category"]
)

print(json.dumps(
    humanitarian_evidence,
    indent=2
))

{
  "supports_disaster_assessment": true,
  "status": "SUPPORTING"
}


CONFLICT-07 — Build conflict assessment

In [8]:
conflict_assessment = {
    "incident_id": evidence_record["incident_id"],
    "evidence_id": evidence_record["evidence_id"],

    "severity_conflict": severity_conflict,

    "temporal_tension": temporal_tension,

    "humanitarian_evidence": humanitarian_evidence,

    "overall_conflict": (
        severity_conflict["conflict"]
    )
}

print(json.dumps(
    conflict_assessment,
    indent=2,
    ensure_ascii=False
))

{
  "incident_id": "CW-001",
  "evidence_id": "EV-001",
  "severity_conflict": {
    "conflict": false,
    "status": "INSUFFICIENT_EVIDENCE",
    "difference": null
  },
  "temporal_tension": {
    "tension": false,
    "status": "CONSISTENT"
  },
  "humanitarian_evidence": {
    "supports_disaster_assessment": true,
    "status": "SUPPORTING"
  },
  "overall_conflict": false
}


CONFLICT-08 — Controlled conflict test

In [9]:
# Controlled test case
# Used only to verify that conflict detection works.

test_conflict = detect_severity_conflict(
    "Low",
    "High"
)

print("CONTROLLED CONFLICT TEST")
print("------------------------")

print(json.dumps(
    test_conflict,
    indent=2
))

CONTROLLED CONFLICT TEST
------------------------
{
  "conflict": true,
  "status": "MAJOR_CONFLICT",
  "difference": 2
}


CONFLICT-09 — Save conflict results

In [10]:
conflict_json_path = os.path.join(
    CONFLICT_RESULTS_DIR,
    "CrisisWeave_Conflict_Assessment_CW001_EV001.json"
)

with open(
    conflict_json_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        conflict_assessment,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Conflict assessment saved:")
print(conflict_json_path)

Conflict assessment saved:
/content/gdrive/MyDrive/CrisisWeave_Project/04_Results/Conflict/CrisisWeave_Conflict_Assessment_CW001_EV001.json


CONFLICT-10 — Complete Temporal evidence

In [12]:
# CONFLICT-10 — Load saved Temporal state and complete temporal evidence

import os
import json

PROJECT_ROOT = "/content/gdrive/MyDrive/CrisisWeave_Project"

# Load saved Temporal state
temporal_path = os.path.join(
    PROJECT_ROOT,
    "04_Results",
    "Temporal",
    "CrisisWeave_Temporal_State_CW001.json"
)

with open(temporal_path, "r", encoding="utf-8") as f:
    temporal_state = json.load(f)

print("Temporal state loaded successfully.")

# Add latest severity to evidence
evidence_record["temporal_evidence"]["latest_severity"] = (
    temporal_state["latest_severity"]
)

print("\nTemporal evidence updated:")
print(json.dumps(
    evidence_record["temporal_evidence"],
    indent=2
))

Temporal state loaded successfully.

Temporal evidence updated:
{
  "severity_trend": "INCREASING",
  "water_extent_trend": "INCREASING",
  "affected_population_trend": "INCREASING",
  "overall_temporal_trend": "INCREASING",
  "progression_direction": "NE",
  "latest_severity": "High"
}


In [13]:
# CONFLICT-11 — Recalculate severity conflict

claims["temporal_latest_severity"] = (
    evidence_record["temporal_evidence"]["latest_severity"]
)

severity_conflict = detect_severity_conflict(
    claims["cnn_severity"],
    claims["temporal_latest_severity"]
)

conflict_assessment["severity_conflict"] = severity_conflict

conflict_assessment["overall_conflict"] = (
    severity_conflict["conflict"]
)

print(json.dumps(
    conflict_assessment,
    indent=2,
    ensure_ascii=False
))

{
  "incident_id": "CW-001",
  "evidence_id": "EV-001",
  "severity_conflict": {
    "conflict": false,
    "status": "AGREEMENT",
    "difference": 0
  },
  "temporal_tension": {
    "tension": false,
    "status": "CONSISTENT"
  },
  "humanitarian_evidence": {
    "supports_disaster_assessment": true,
    "status": "SUPPORTING"
  },
  "overall_conflict": false
}


In [14]:
# CONFLICT-12 — Save corrected evidence and conflict assessment

import os
import json

PROJECT_ROOT = "/content/gdrive/MyDrive/CrisisWeave_Project"

# Save corrected unified evidence
evidence_path = os.path.join(
    PROJECT_ROOT,
    "04_Results",
    "Evidence",
    "CrisisWeave_Evidence_CW001_EV001.json"
)

with open(
    evidence_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        evidence_record,
        f,
        indent=2,
        ensure_ascii=False
    )


# Save conflict assessment
conflict_json_path = os.path.join(
    PROJECT_ROOT,
    "04_Results",
    "Conflict",
    "CrisisWeave_Conflict_Assessment_CW001_EV001.json"
)

with open(
    conflict_json_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        conflict_assessment,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Corrected evidence saved:")
print(evidence_path)

print("\nConflict assessment saved:")
print(conflict_json_path)

Corrected evidence saved:
/content/gdrive/MyDrive/CrisisWeave_Project/04_Results/Evidence/CrisisWeave_Evidence_CW001_EV001.json

Conflict assessment saved:
/content/gdrive/MyDrive/CrisisWeave_Project/04_Results/Conflict/CrisisWeave_Conflict_Assessment_CW001_EV001.json
